In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("Raw directory exists:", RAW_DIR.exists())
print("Processed directory exists:", PROCESSED_DIR.exists())

Project root: c:\Users\KIIT\Desktop\AI-Disaster-Intelligence-System
Raw directory exists: True
Processed directory exists: True


In [2]:
csv_files = sorted(
    list(RAW_DIR.glob("*.csv")) +
    list(PROCESSED_DIR.glob("*.csv"))
)

print(f"Found {len(csv_files)} CSV file(s):\n")

for file in csv_files:
    print("-", file.relative_to(PROJECT_ROOT))

Found 5 CSV file(s):

- data\processed\flood_features.csv
- data\processed\flood_labeled_clean.csv
- data\processed\test.csv
- data\processed\train.csv
- data\raw\FloodPrediction.csv


In [3]:
for file in csv_files:
    df = pd.read_csv(file)

    missing = df.isna().sum()
    missing = missing[missing > 0]

    print("=" * 80)
    print(f"Dataset: {file.relative_to(PROJECT_ROOT)}")

    if missing.empty:
        print("No missing values found.")
    else:
        missing_report = pd.DataFrame({
            "missing_count": missing,
            "missing_percentage": (missing / len(df) * 100).round(2)
        })

        display(missing_report)

Dataset: data\processed\flood_features.csv
No missing values found.
Dataset: data\processed\flood_labeled_clean.csv
No missing values found.
Dataset: data\processed\test.csv
No missing values found.
Dataset: data\processed\train.csv
No missing values found.
Dataset: data\raw\FloodPrediction.csv


,missing_count,missing_percentage
Flood?,16051,78.13


In [4]:
duplicate_records = []

for file in csv_files:
    df = pd.read_csv(file)

    duplicate_count = int(df.duplicated().sum())

    duplicate_records.append({
        "dataset": str(file.relative_to(PROJECT_ROOT)),
        "rows": len(df),
        "duplicate_rows": duplicate_count,
        "duplicate_percentage": round(
            duplicate_count / len(df) * 100, 2
        )
    })

duplicate_report = pd.DataFrame(duplicate_records)

display(duplicate_report)

,dataset,rows,duplicate_rows,duplicate_percentage
0,data\processed\flood_features.csv,4493,0,0.0
1,data\processed\flood_labeled_clean.csv,4493,0,0.0
2,data\processed\test.csv,474,0,0.0
3,data\processed\train.csv,4019,0,0.0
4,data\raw\FloodPrediction.csv,20544,0,0.0


In [5]:
feature_duplicate_records = []

for file in csv_files:
    df = pd.read_csv(file)

    # Identify possible target columns using a simple reusable heuristic
    target_candidates = []

    for col in df.columns:
        unique_count = df[col].nunique(dropna=True)

        if 2 <= unique_count <= 10:
            target_candidates.append(col)

    # Exclude candidate targets from the feature comparison
    feature_columns = [
        col for col in df.columns
        if col not in target_candidates
    ]

    duplicate_feature_count = int(
        df.duplicated(subset=feature_columns).sum()
    )

    feature_duplicate_records.append({
        "dataset": str(file.relative_to(PROJECT_ROOT)),
        "feature_columns_used": len(feature_columns),
        "duplicate_feature_rows": duplicate_feature_count,
        "duplicate_feature_percentage": round(
            duplicate_feature_count / len(df) * 100, 2
        )
    })

feature_duplicate_report = pd.DataFrame(feature_duplicate_records)

display(feature_duplicate_report)

,dataset,feature_columns_used,duplicate_feature_rows,duplicate_feature_percentage
0,data\processed\flood_features.csv,15,0,0.0
1,data\processed\flood_labeled_clean.csv,18,0,0.0
2,data\processed\test.csv,14,0,0.0
3,data\processed\train.csv,15,0,0.0
4,data\raw\FloodPrediction.csv,18,0,0.0


In [6]:
constant_records = []

for file in csv_files:
    df = pd.read_csv(file)

    constant_columns = [
        col for col in df.columns
        if df[col].nunique(dropna=False) <= 1
    ]

    constant_records.append({
        "dataset": str(file.relative_to(PROJECT_ROOT)),
        "constant_column_count": len(constant_columns),
        "constant_columns": constant_columns
    })

constant_report = pd.DataFrame(constant_records)

display(constant_report)

,dataset,constant_column_count,constant_columns
0,data\processed\flood_features.csv,0,[]
1,data\processed\flood_labeled_clean.csv,0,[]
2,data\processed\test.csv,0,[]
3,data\processed\train.csv,0,[]
4,data\raw\FloodPrediction.csv,0,[]


In [7]:
near_constant_records = []

DOMINANCE_THRESHOLD = 0.99

for file in csv_files:
    df = pd.read_csv(file)

    near_constant_columns = []

    for col in df.columns:
        value_counts = df[col].value_counts(dropna=False)

        if len(value_counts) == 0:
            continue

        dominant_fraction = value_counts.iloc[0] / len(df)

        if dominant_fraction >= DOMINANCE_THRESHOLD:
            near_constant_columns.append({
                "column": col,
                "dominant_value": value_counts.index[0],
                "dominant_fraction": round(dominant_fraction, 4)
            })

    for item in near_constant_columns:
        near_constant_records.append({
            "dataset": str(file.relative_to(PROJECT_ROOT)),
            **item
        })

near_constant_report = pd.DataFrame(near_constant_records)

if near_constant_report.empty:
    print("No near-constant columns found at the 99% dominance threshold.")
else:
    display(near_constant_report)

No near-constant columns found at the 99% dominance threshold.


In [8]:
dtype_records = []

for file in csv_files:
    df = pd.read_csv(file)

    for col in df.columns:
        dtype_records.append({
            "dataset": str(file.relative_to(PROJECT_ROOT)),
            "column": col,
            "dtype": str(df[col].dtype),
            "non_null": int(df[col].notna().sum()),
            "unique_values": int(df[col].nunique(dropna=True))
        })

dtype_report = pd.DataFrame(dtype_records)

display(dtype_report)

,dataset,column,dtype,non_null,unique_values
0,data\processed\flood_features.csv,Station_Names,str,4493,33
1,data\processed\flood_features.csv,Year,int64,4493,66
2,data\processed\flood_features.csv,Month,int64,4493,12
3,data\processed\flood_features.csv,Max_Temp,float64,4493,151
4,data\processed\flood_features.csv,Min_Temp,float64,4493,182
...,...,...,...,...,...
87,data\raw\FloodPrediction.csv,LATITUDE,float64,20544,30
88,data\raw\FloodPrediction.csv,LONGITUDE,float64,20544,33
89,data\raw\FloodPrediction.csv,ALT,int64,20544,17
90,data\raw\FloodPrediction.csv,Period,float64,20544,792


In [9]:
for file in csv_files:
    df = pd.read_csv(file)

    text_columns = df.select_dtypes(include=["object", "string"]).columns.tolist()

    print("=" * 80)
    print(f"Dataset: {file.relative_to(PROJECT_ROOT)}")
    print("Text/categorical columns:")
    print(text_columns)

Dataset: data\processed\flood_features.csv
Text/categorical columns:
['Station_Names', 'Season', 'Rainfall_Category']
Dataset: data\processed\flood_labeled_clean.csv
Text/categorical columns:
['Station_Names']
Dataset: data\processed\test.csv
Text/categorical columns:
['Station_Names', 'Season', 'Rainfall_Category']
Dataset: data\processed\train.csv
Text/categorical columns:
['Station_Names', 'Season', 'Rainfall_Category']
Dataset: data\raw\FloodPrediction.csv
Text/categorical columns:
['Station_Names']


In [10]:
for file in csv_files:
    df = pd.read_csv(file)

    text_columns = df.select_dtypes(
        include=["object", "string"]
    ).columns.tolist()

    print("=" * 80)
    print(f"Dataset: {file.relative_to(PROJECT_ROOT)}")

    for col in text_columns:
        values = df[col].dropna().unique()

        print(f"\n{col} ({len(values)} unique values):")
        print(sorted(map(str, values)))

Dataset: data\processed\flood_features.csv

Station_Names (33 unique values):
['Barisal', 'Bhola', 'Bogra', 'Chandpur', 'Chittagong (City-Ambagan)', 'Chittagong (IAP-Patenga)', 'Comilla', "Cox's Bazar", 'Dhaka', 'Dinajpur', 'Faridpur', 'Feni', 'Hatiya', 'Ishurdi', 'Jessore', 'Khepupara', 'Khulna', 'Kutubdia', 'Madaripur', 'Maijdee Court', 'Mongla', 'Mymensingh', 'Patuakhali', 'Rajshahi', 'Rangamati', 'Rangpur', 'Sandwip', 'Satkhira', 'Sitakunda', 'Srimangal', 'Sylhet', 'Tangail', 'Teknaf']

Season (4 unique values):
['Monsoon', 'Post-Monsoon', 'Pre-Monsoon', 'Winter']

Rainfall_Category (4 unique values):
['Extreme', 'High', 'Low', 'Moderate']
Dataset: data\processed\flood_labeled_clean.csv

Station_Names (33 unique values):
['Barisal', 'Bhola', 'Bogra', 'Chandpur', 'Chittagong (City-Ambagan)', 'Chittagong (IAP-Patenga)', 'Comilla', "Cox's Bazar", 'Dhaka', 'Dinajpur', 'Faridpur', 'Feni', 'Hatiya', 'Ishurdi', 'Jessore', 'Khepupara', 'Khulna', 'Kutubdia', 'Madaripur', 'Maijdee Court', 'M

## 5. Numerical Range and Invalid Value Checks

This section checks numerical columns for potentially invalid or suspicious values.

Values are not automatically modified. Any unusual values will be documented for review before making changes to the ML pipeline.

In [ ]:
# Check numerical ranges for each dataset

for file in csv_files:
    df = pd.read_csv(file)

    print("=" * 80)
    print(f"Dataset: {file.relative_to(PROJECT_ROOT)}")

    numeric_cols = df.select_dtypes(include="number").columns

    range_table = pd.DataFrame({
        "min": df[numeric_cols].min(),
        "max": df[numeric_cols].max(),
        "mean": df[numeric_cols].mean(),
        "median": df[numeric_cols].median()
    })

    display(range_table)

Dataset: data\raw\FloodPrediction.csv


,min,max,mean,median
Sl,0.00,20543.00,10271.500000,10271.50
Year,1948.00,2013.00,1985.332944,1987.00
Month,1.00,12.00,6.500000,6.50
Max_Temp,21.60,44.00,33.450739,33.90
Min_Temp,6.20,28.10,21.166872,23.40
Rainfall,0.00,2072.00,198.776621,111.00
Relative_Humidity,34.00,97.00,79.497375,81.00
Wind_Speed,0.00,11.20,1.415049,1.20
Cloud_Coverage,0.00,7.90,3.485827,3.30
Bright_Sunshine,0.00,11.00,6.419056,6.80


Dataset: data\processed\flood_features.csv


,min,max,mean,median
Year,1948.00,2013.00,1986.393056,1988.000000
Month,1.00,12.00,7.221678,7.000000
Max_Temp,26.00,43.70,34.017633,34.000000
Min_Temp,8.50,27.70,24.589457,25.400000
Rainfall,0.00,2072.00,526.001857,494.000000
Relative_Humidity,52.00,96.00,85.363425,87.000000
Wind_Speed,0.00,7.80,1.792189,1.544444
Cloud_Coverage,0.00,7.90,5.554604,6.000000
Bright_Sunshine,0.70,10.70,4.736064,4.435849
LATITUDE,20.87,25.72,23.237650,22.830000


Dataset: data\processed\flood_labeled_clean.csv


,min,max,mean,median
Sl,5.00,20539.00,9724.902292,8480.000000
Year,1948.00,2013.00,1986.393056,1988.000000
Month,1.00,12.00,7.221678,7.000000
Max_Temp,26.00,43.70,34.017633,34.000000
Min_Temp,8.50,27.70,24.589457,25.400000
Rainfall,0.00,2072.00,526.001857,494.000000
Relative_Humidity,52.00,96.00,85.363425,87.000000
Wind_Speed,0.00,7.80,1.792189,1.544444
Cloud_Coverage,0.00,7.90,5.554604,6.000000
Bright_Sunshine,0.70,10.70,4.736064,4.435849


Dataset: data\processed\test.csv


,min,max,mean,median
Year,2009.00,2013.00,2011.149789,2011.00
Month,1.00,12.00,7.227848,7.00
Max_Temp,26.00,40.30,34.635513,34.90
Min_Temp,8.50,27.60,24.490577,25.60
Rainfall,0.00,1688.00,489.214487,466.50
Relative_Humidity,52.00,95.00,83.872011,86.00
Wind_Speed,0.10,7.80,2.612025,2.40
Cloud_Coverage,0.10,7.50,5.340506,6.00
Bright_Sunshine,1.60,9.10,4.918074,4.60
LATITUDE,20.87,25.72,23.055907,22.70


Dataset: data\processed\train.csv


,min,max,mean,median
Year,1948.00,2008.00,1983.473252,1985.000000
Month,1.00,12.00,7.220950,7.000000
Max_Temp,26.30,43.70,33.944761,34.000000
Min_Temp,9.50,27.70,24.601119,25.300000
Rainfall,0.00,2072.00,530.340552,497.000000
Relative_Humidity,54.00,96.00,85.539322,87.000000
Wind_Speed,0.00,7.30,1.695498,1.424074
Cloud_Coverage,0.00,7.90,5.579854,6.000000
Bright_Sunshine,0.70,10.70,4.714598,4.424138
LATITUDE,20.87,25.72,23.259084,22.830000


In [12]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

datasets = {}

csv_files = list((PROJECT_ROOT / "data" / "raw").glob("*.csv"))
csv_files += list((PROJECT_ROOT / "data" / "processed").glob("*.csv"))

for file_path in csv_files:
    datasets[str(file_path.relative_to(PROJECT_ROOT))] = pd.read_csv(file_path)

print(f"Loaded {len(datasets)} datasets:")
for name, df in datasets.items():
    print(f"- {name}: {df.shape}")

Loaded 5 datasets:
- data\raw\FloodPrediction.csv: (20544, 19)
- data\processed\flood_features.csv: (4493, 18)
- data\processed\flood_labeled_clean.csv: (4493, 19)
- data\processed\test.csv: (474, 18)
- data\processed\train.csv: (4019, 18)


In [14]:
# Define expected logical ranges for selected columns

expected_ranges = {
    "Month": (1, 12),
    "Max_Temp": (-50, 60),
    "Min_Temp": (-50, 60),
    "Relative_Humidity": (0, 100),
    "LATITUDE": (-90, 90),
    "LONGITUDE": (-180, 180),
    "ALT": (-500, 9000),
    "Flood?": (0, 1)
}

invalid_results = []

for name, df in datasets.items():
    for column, (lower, upper) in expected_ranges.items():
        if column in df.columns:
            invalid_count = ((df[column] < lower) | (df[column] > upper)).sum()
            
            invalid_results.append({
                "dataset": name,
                "column": column,
                "expected_range": f"{lower} to {upper}",
                "invalid_count": int(invalid_count)
            })

invalid_df = pd.DataFrame(invalid_results)

display(invalid_df)

,dataset,column,expected_range,invalid_count
0,data\raw\FloodPrediction.csv,Month,1 to 12,0
1,data\raw\FloodPrediction.csv,Max_Temp,-50 to 60,0
2,data\raw\FloodPrediction.csv,Min_Temp,-50 to 60,0
3,data\raw\FloodPrediction.csv,Relative_Humidity,0 to 100,0
4,data\raw\FloodPrediction.csv,LATITUDE,-90 to 90,0
5,data\raw\FloodPrediction.csv,LONGITUDE,-180 to 180,0
6,data\raw\FloodPrediction.csv,ALT,-500 to 9000,0
7,data\raw\FloodPrediction.csv,Flood?,0 to 1,0
8,data\processed\flood_features.csv,Month,1 to 12,0
9,data\processed\flood_features.csv,Max_Temp,-50 to 60,0


## 7. Potential Identifier Analysis

This section investigates columns that may function as identifiers rather than meaningful predictive features.

Identifier-like columns can cause problems if they contain information that does not generalize to future observations.

No columns will be removed automatically.

In [15]:
# Investigate potential identifier-like columns

identifier_candidates = [
    "Sl",
    "Station_Number",
    "Period"
]

identifier_results = []

for name, df in datasets.items():
    for column in identifier_candidates:
        if column in df.columns:
            identifier_results.append({
                "dataset": name,
                "column": column,
                "rows": len(df),
                "unique_values": df[column].nunique(),
                "unique_percentage": round(
                    df[column].nunique() / len(df) * 100, 2
                ),
                "first_values": df[column].head(5).tolist()
            })

identifier_df = pd.DataFrame(identifier_results)

display(identifier_df)

,dataset,column,rows,unique_values,unique_percentage,first_values
0,data\raw\FloodPrediction.csv,Sl,20544,20544,100.00,"[0, 1, 2, 3, 4]"
1,data\raw\FloodPrediction.csv,Station_Number,20544,33,0.16,"[41950, 41950, 41950, 41950, 41950]"
2,data\raw\FloodPrediction.csv,Period,20544,792,3.86,"[1949.01, 1949.02, 1949.03, 1949.04, 1949.05]"
3,data\processed\flood_labeled_clean.csv,Sl,4493,4493,100.00,"[5, 6, 7, 16, 17]"
4,data\processed\flood_labeled_clean.csv,Station_Number,4493,33,0.73,"[41950, 41950, 41950, 41950, 41950]"
5,data\processed\flood_labeled_clean.csv,Period,4493,664,14.78,"[1949.06, 1949.07, 1949.08, 1950.05, 1950.06]"


## 8. Station Identifier Consistency

This section checks whether each Station_Number corresponds to exactly one Station_Name and geographic location.

This helps determine whether Station_Number is simply an identifier or contains information that differs from the explicit station/location features.

In [16]:
# Check consistency between Station_Number and station/location information

for name, df in datasets.items():
    if "Station_Number" not in df.columns:
        continue

    print("=" * 80)
    print(f"Dataset: {name}")

    station_mapping = (
        df.groupby("Station_Number")
        .agg(
            station_names=("Station_Names", "nunique") if "Station_Names" in df.columns else ("Station_Number", "size"),
            latitudes=("LATITUDE", "nunique") if "LATITUDE" in df.columns else ("Station_Number", "size"),
            longitudes=("LONGITUDE", "nunique") if "LONGITUDE" in df.columns else ("Station_Number", "size"),
            altitudes=("ALT", "nunique") if "ALT" in df.columns else ("Station_Number", "size")
        )
        .reset_index()
    )

    display(station_mapping)

Dataset: data\raw\FloodPrediction.csv


,Station_Number,station_names,latitudes,longitudes,altitudes
0,41859,1,1,1,1
1,41863,1,1,1,1
2,41883,1,1,1,1
3,41886,1,1,1,1
4,41891,1,1,1,1
5,41895,1,1,1,1
6,41907,1,1,1,1
7,41909,1,1,1,1
8,41915,1,1,1,1
9,41923,1,1,1,1


Dataset: data\processed\flood_labeled_clean.csv


,Station_Number,station_names,latitudes,longitudes,altitudes
0,41859,1,1,1,1
1,41863,1,1,1,1
2,41883,1,1,1,1
3,41886,1,1,1,1
4,41891,1,1,1,1
5,41895,1,1,1,1
6,41907,1,1,1,1
7,41909,1,1,1,1
8,41915,1,1,1,1
9,41923,1,1,1,1


## Identifier Analysis — Findings

### Sl
`Sl` is 100% unique in the raw dataset and the labeled dataset. This indicates that it functions as a row-level identifier rather than an environmental feature.

### Station_Number
`Station_Number` contains 33 unique values and each value maps consistently to exactly one:

- Station_Name
- Latitude
- Longitude
- Altitude

Therefore, `Station_Number` appears to be a station identifier and is potentially redundant with the explicit station and geographic features.

### Period
`Period` represents the year and month in a numeric format such as `1949.01`, `1949.02`, etc. It should be considered a temporal representation rather than a simple row identifier.

### Recommendation
No columns are removed at this stage. These findings should be discussed with the Technical Lead before making any changes to the ML feature set.

## 9. Outlier Analysis

This section identifies statistically extreme values using the Interquartile Range (IQR) method.

An outlier identified by this method is not automatically considered an invalid value. Extreme weather observations may represent genuine events and can be particularly important for flood prediction.

Therefore, outliers will be reported for investigation rather than automatically removed.

In [17]:
# IQR-based outlier analysis

outlier_results = []

for name, df in datasets.items():
    numeric_cols = df.select_dtypes(include="number").columns

    for column in numeric_cols:
        # Skip binary target
        if column == "Flood?":
            continue

        q1 = df[column].quantile(0.25)
        q3 = df[column].quantile(0.75)
        iqr = q3 - q1

        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        outlier_count = (
            (df[column] < lower_bound) |
            (df[column] > upper_bound)
        ).sum()

        outlier_results.append({
            "dataset": name,
            "column": column,
            "outlier_count": int(outlier_count),
            "outlier_percentage": round(
                outlier_count / len(df) * 100, 2
            ),
            "lower_bound": round(lower_bound, 2),
            "upper_bound": round(upper_bound, 2)
        })

outlier_df = pd.DataFrame(outlier_results)

display(
    outlier_df
    .sort_values(
        ["dataset", "outlier_percentage"],
        ascending=[True, False]
    )
)

,dataset,column,outlier_count,outlier_percentage,lower_bound,upper_bound
20,data\processed\flood_features.csv,Min_Temp,422,9.39,22.80,27.60
24,data\processed\flood_features.csv,Cloud_Coverage,405,9.01,3.35,8.29
22,data\processed\flood_features.csv,Relative_Humidity,339,7.55,78.00,94.00
29,data\processed\flood_features.csv,Temperature_Range,218,4.85,4.15,14.15
18,data\processed\flood_features.csv,Month,209,4.65,3.00,11.00
...,...,...,...,...,...,...
8,data\raw\FloodPrediction.csv,Cloud_Coverage,0,0.00,-4.25,11.35
10,data\raw\FloodPrediction.csv,Station_Number,0,0.00,41828.00,42044.00
13,data\raw\FloodPrediction.csv,LATITUDE,0,0.00,20.17,26.76
14,data\raw\FloodPrediction.csv,LONGITUDE,0,0.00,86.68,94.32


## Outlier Analysis — Initial Findings

The IQR method identified statistical outliers in several numerical variables.

The presence of an IQR outlier does not necessarily indicate an invalid observation. Weather variables can naturally contain extreme values, particularly during unusual weather events.

The `Month` variable was also flagged by the IQR method. This is a limitation of applying a generic numerical outlier method to a bounded cyclical variable. Values between 1 and 12 are logically valid, so these observations should not be treated as invalid.

Variables such as rainfall, temperature, humidity, and derived weather features require domain-based interpretation before any outlier treatment is considered.

No outliers are removed or modified at this stage.

## 10. Extreme Value Inspection

This section examines the most extreme observations for selected weather variables.

The purpose is to determine whether statistically extreme values appear plausible or potentially erroneous.

No values are modified.

In [18]:
# Inspect extreme values in important weather variables

variables_to_inspect = [
    "Rainfall",
    "Max_Temp",
    "Min_Temp",
    "Relative_Humidity",
    "Wind_Speed",
    "Cloud_Coverage",
    "Bright_Sunshine"
]

df = datasets["data\\raw\\FloodPrediction.csv"]

for column in variables_to_inspect:
    if column in df.columns:
        print("=" * 80)
        print(f"{column} — Highest 10 values")
        
        display(
            df.nlargest(10, column)[
                ["Station_Names", "Year", "Month", column, "Flood?"]
            ]
        )

Rainfall — Highest 10 values


,Station_Names,Year,Month,Rainfall,Flood?
20478,Teknaf,2008,7,2072.0,1.0
4842,Cox's Bazar,1987,7,1885.0,1.0
16940,Sandwip,2004,9,1826.0,1.0
16901,Sandwip,2001,6,1784.0,1.0
20393,Teknaf,2001,6,1759.0,1.0
16914,Sandwip,2002,7,1698.0,1.0
11670,Kutubdia,2012,7,1688.0,1.0
2813,Chittagong (IAP-Patenga),1950,6,1676.0,1.0
11502,Kutubdia,1998,7,1675.0,1.0
20454,Teknaf,2006,7,1650.0,1.0


Max_Temp — Highest 10 values


,Station_Names,Year,Month,Max_Temp,Flood?
8656,Ishurdi,1970,5,44.0,NaN
1480,Bogra,1958,5,43.9,NaN
14727,Rajshahi,1985,4,43.8,NaN
1623,Bogra,1970,4,43.7,NaN
14776,Rajshahi,1989,5,43.7,1.0
1624,Bogra,1970,5,43.6,NaN
1695,Bogra,1976,4,43.6,NaN
8571,Ishurdi,1963,4,43.6,NaN
8716,Ishurdi,1975,5,43.6,NaN
1478,Bogra,1958,3,43.5,NaN


Min_Temp — Highest 10 values


...

,Station_Names,Year,Month,Wind_Speed,Flood?
9964,Jessore,2013,5,11.2,NaN
9927,Jessore,2010,4,9.6,NaN
2210,Chandpur,1969,3,9.3,NaN
783,Bhola,1966,4,9.1,NaN
9950,Jessore,2012,3,9.0,NaN
9938,Jessore,2011,3,8.6,NaN
9953,Jessore,2012,6,8.6,NaN
9915,Jessore,2009,4,8.3,NaN
9941,Jessore,2011,6,8.0,NaN
9954,Jessore,2012,7,8.0,NaN


Cloud_Coverage — Highest 10 values


,Station_Names,Year,Month,Cloud_Coverage,Flood?
3751,Comilla,1962,8,7.9,NaN
13219,Mymensingh,1950,8,7.9,1.0
211,Barisal,1966,8,7.8,1.0
822,Bhola,1969,7,7.7,1.0
19157,Sylhet,1962,6,7.7,1.0
19302,Sylhet,1974,7,7.7,1.0
19471,Sylhet,1988,8,7.7,1.0
809,Bhola,1968,6,7.6,NaN
823,Bhola,1969,8,7.6,1.0
3079,Chittagong (IAP-Patenga),1972,8,7.6,1.0


Bright_Sunshine — Highest 10 values


,Station_Names,Year,Month,Bright_Sunshine,Flood?
20149,Teknaf,1981,2,11.0,NaN
4706,Cox's Bazar,1976,3,10.9,NaN
11518,Kutubdia,1999,11,10.9,NaN
301,Barisal,1974,2,10.8,NaN
4645,Cox's Bazar,1971,2,10.8,NaN
5017,Cox's Bazar,2002,2,10.8,NaN
16089,Rangpur,1981,10,10.8,NaN
1694,Bogra,1976,3,10.7,NaN
1730,Bogra,1979,3,10.7,NaN
4681,Cox's Bazar,1974,2,10.7,NaN


In [19]:
# ============================================================
# TARGET VARIABLE ANALYSIS
# ============================================================

# Use the cleaned labeled dataset
df = datasets["data\\processed\\flood_labeled_clean.csv"].copy()

print("=" * 80)
print("TARGET VARIABLE ANALYSIS")
print("=" * 80)

# 1. Target counts
target_counts = df["Flood?"].value_counts().sort_index()

print("\nFlood label counts:")
print(target_counts)

# 2. Target percentages
target_percentages = (
    df["Flood?"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

print("\nFlood label percentages:")
print(target_percentages)

# 3. Combined summary
target_summary = pd.DataFrame({
    "count": target_counts,
    "percentage": target_percentages
})

print("\nTarget distribution:")
display(target_summary)

# 4. Basic interpretation
print("\nInterpretation:")

for label in target_counts.index:
    print(
        f"Flood = {label}: "
        f"{target_counts[label]} rows "
        f"({target_percentages[label]:.2f}%)"
    )

# 5. Check whether the target is imbalanced
max_percentage = target_percentages.max()
min_percentage = target_percentages.min()

if max_percentage >= 70:
    print("\nWARNING: The target is strongly imbalanced.")
elif max_percentage >= 60:
    print("\nThe target shows moderate class imbalance.")
else:
    print("\nThe target is relatively balanced.")

TARGET VARIABLE ANALYSIS

Flood label counts:
Flood?
0     361
1    4132
Name: count, dtype: int64

Flood label percentages:
Flood?
0     8.03
1    91.97
Name: proportion, dtype: float64

Target distribution:


,count,percentage
Flood?,,
0,361,8.03
1,4132,91.97



Interpretation:
Flood = 0: 361 rows (8.03%)
Flood = 1: 4132 rows (91.97%)



In [20]:
# ============================================================
# FLOOD DISTRIBUTION BY STATION, MONTH AND SEASON
# ============================================================

df = datasets["data\\processed\\flood_labeled_clean.csv"].copy()

# ------------------------------------------------------------
# 1. Flood distribution by station
# ------------------------------------------------------------

station_summary = (
    df.groupby("Station_Names")["Flood?"]
    .agg(
        total_records="count",
        flood_count="sum",
        flood_rate="mean"
    )
    .sort_values("flood_rate", ascending=False)
)

station_summary["flood_rate"] = (
    station_summary["flood_rate"] * 100
).round(2)

print("=" * 80)
print("FLOOD DISTRIBUTION BY STATION")
print("=" * 80)

display(station_summary)


# ------------------------------------------------------------
# 2. Flood distribution by month
# ------------------------------------------------------------

month_summary = (
    df.groupby("Month")["Flood?"]
    .agg(
        total_records="count",
        flood_count="sum",
        flood_rate="mean"
    )
)

month_summary["flood_rate"] = (
    month_summary["flood_rate"] * 100
).round(2)

print("\n" + "=" * 80)
print("FLOOD DISTRIBUTION BY MONTH")
print("=" * 80)

display(month_summary)


# ------------------------------------------------------------
# 3. Flood distribution by season
# ------------------------------------------------------------

# Season may not exist in this cleaned dataset.
# Create it from Month if necessary.

if "Season" not in df.columns:

    def get_season(month):
        if month in [3, 4, 5]:
            return "Pre-Monsoon"
        elif month in [6, 7, 8, 9]:
            return "Monsoon"
        elif month in [10, 11]:
            return "Post-Monsoon"
        else:
            return "Winter"

    df["Season"] = df["Month"].apply(get_season)


season_summary = (
    df.groupby("Season")["Flood?"]
    .agg(
        total_records="count",
        flood_count="sum",
        flood_rate="mean"
    )
)

season_summary["flood_rate"] = (
    season_summary["flood_rate"] * 100
).round(2)

print("\n" + "=" * 80)
print("FLOOD DISTRIBUTION BY SEASON")
print("=" * 80)

display(season_summary)

FLOOD DISTRIBUTION BY STATION


,total_records,flood_count,flood_rate
Station_Names,,,
Barisal,235,235,100.00
Bhola,104,104,100.00
Bogra,107,107,100.00
Chandpur,130,130,100.00
Chittagong (City-Ambagan),15,15,100.00
Chittagong (IAP-Patenga),201,201,100.00
Comilla,146,146,100.00
Feni,140,140,100.00
Faridpur,87,87,100.00



FLOOD DISTRIBUTION BY MONTH


,total_records,flood_count,flood_rate
Month,,,
1,69,0,0.00
2,69,0,0.00
3,7,4,57.14
4,49,46,93.88
5,320,318,99.38
6,964,963,99.90
7,1131,1130,99.91
8,881,880,99.89
9,576,574,99.65



FLOOD DISTRIBUTION BY SEASON


,total_records,flood_count,flood_rate
Season,,,
Monsoon,3552,3547,99.86
Post-Monsoon,356,217,60.96
Pre-Monsoon,376,368,97.87
Winter,209,0,0.00


In [21]:
# ============================================================
# LABELED vs UNLABELED DATA ANALYSIS
# ============================================================

raw_df = datasets["data\\raw\\FloodPrediction.csv"].copy()
labeled_df = datasets["data\\processed\\flood_labeled_clean.csv"].copy()

print("=" * 80)
print("LABELED vs UNLABELED DATA ANALYSIS")
print("=" * 80)

# ------------------------------------------------------------
# 1. Basic counts
# ------------------------------------------------------------

labeled_count = raw_df["Flood?"].notna().sum()
unlabeled_count = raw_df["Flood?"].isna().sum()

print(f"\nTotal raw records      : {len(raw_df)}")
print(f"Labeled records        : {labeled_count}")
print(f"Unlabeled records      : {unlabeled_count}")
print(
    f"Labeled percentage     : "
    f"{labeled_count / len(raw_df) * 100:.2f}%"
)


# ------------------------------------------------------------
# 2. Compare months
# ------------------------------------------------------------

raw_labeled = raw_df[raw_df["Flood?"].notna()].copy()
raw_unlabeled = raw_df[raw_df["Flood?"].isna()].copy()

print("\n" + "=" * 80)
print("MONTH DISTRIBUTION: LABELED vs UNLABELED")
print("=" * 80)

labeled_month = (
    raw_labeled["Month"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

unlabeled_month = (
    raw_unlabeled["Month"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

month_comparison = pd.DataFrame({
    "labeled_%": labeled_month,
    "unlabeled_%": unlabeled_month
}).fillna(0)

display(month_comparison)


# ------------------------------------------------------------
# 3. Compare stations
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("STATION DISTRIBUTION: LABELED vs UNLABELED")
print("=" * 80)

labeled_station = (
    raw_labeled["Station_Names"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

unlabeled_station = (
    raw_unlabeled["Station_Names"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

station_comparison = pd.DataFrame({
    "labeled_%": labeled_station,
    "unlabeled_%": unlabeled_station
}).fillna(0)

display(station_comparison)


# ------------------------------------------------------------
# 4. Compare numerical features
# ------------------------------------------------------------

numeric_features = [
    "Max_Temp",
    "Min_Temp",
    "Rainfall",
    "Relative_Humidity",
    "Wind_Speed",
    "Cloud_Coverage",
    "Bright_Sunshine",
    "LATITUDE",
    "LONGITUDE",
    "ALT"
]

print("\n" + "=" * 80)
print("NUMERICAL FEATURE COMPARISON")
print("=" * 80)

comparison_rows = []

for col in numeric_features:

    comparison_rows.append({
        "feature": col,
        "labeled_mean": raw_labeled[col].mean(),
        "unlabeled_mean": raw_unlabeled[col].mean(),
        "labeled_median": raw_labeled[col].median(),
        "unlabeled_median": raw_unlabeled[col].median()
    })

comparison_df = pd.DataFrame(comparison_rows)

display(comparison_df.round(3))

LABELED vs UNLABELED DATA ANALYSIS

Total raw records      : 20544
Labeled records        : 4493
Unlabeled records      : 16051
Labeled percentage     : 21.87%

MONTH DISTRIBUTION: LABELED vs UNLABELED


,labeled_%,unlabeled_%
Month,,
1,1.54,10.24
2,1.54,10.24
3,0.16,10.62
4,1.09,10.36
5,7.12,8.67
6,21.46,4.66
7,25.17,3.62
8,19.61,5.18
9,12.82,7.08



STATION DISTRIBUTION: LABELED vs UNLABELED


,labeled_%,unlabeled_%
Station_Names,,
Barisal,5.23,3.40
Bhola,2.31,2.94
Bogra,2.38,4.27
Chandpur,2.89,2.93
Chittagong (City-Ambagan),0.33,0.21
Chittagong (IAP-Patenga),4.47,3.61
Comilla,3.25,4.02
Cox's Bazar,5.94,3.27
Dhaka,7.19,2.55



NUMERICAL FEATURE COMPARISON


,feature,labeled_mean,unlabeled_mean,labeled_median,unlabeled_median
0,Max_Temp,34.018,33.292,34.000,33.700
1,Min_Temp,24.589,20.209,25.400,21.300
2,Rainfall,526.002,107.180,494.000,52.000
3,Relative_Humidity,85.363,77.855,87.000,79.000
4,Wind_Speed,1.792,1.309,1.544,1.100
5,Cloud_Coverage,5.555,2.907,6.000,2.600
6,Bright_Sunshine,4.736,6.890,4.436,7.171
7,LATITUDE,23.238,23.352,22.830,23.170
8,LONGITUDE,90.656,90.448,90.670,90.390
9,ALT,13.392,13.348,6.000,7.000
